In [13]:
import requests
import pandas as pd 
from sunpy.net import Fido, attrs as a
import dkist.net
import astropy.units as u

In [2]:
url = "https://p3sc.oma.be/api/L3"

params = {
    "select": "DATE-OBS,DATE-END,FILTER,orbit_id",
    "active": "is.true",
    "version": "eq.v03",
    "order": "DATE-OBS.asc",
}

data = requests.get(url, params=params).json()
df = pd.DataFrame(data)

df["DATE-OBS"] = pd.to_datetime(df["DATE-OBS"],format="ISO8601")
df["DATE-END"] = pd.to_datetime(df["DATE-END"],format="ISO8601")

In [6]:
windows = (
    df.groupby(["orbit_id"])
      .agg(
          start=("DATE-OBS", "min"),
          end=("DATE-END", "max"),
          n_images=("DATE-OBS", "size"),
      )
      .reset_index()
      .sort_values("start")
)

In [11]:
windows

,orbit_id,start,end,n_images
0,273,2025-07-16 16:35:06.952,2025-07-16 19:07:08.544,235
1,274,2025-07-17 12:31:08.560,2025-07-17 15:03:08.791,228
2,275,2025-07-18 05:26:16.833,2025-07-18 10:47:51.569,638
3,276,2025-07-19 02:54:09.716,2025-07-19 03:13:54.765,12
4,277,2025-07-19 20:45:16.752,2025-07-20 02:06:50.486,948
...,...,...,...,...
61,760,2026-08-19 14:21:54.477,2026-08-19 18:16:30.123,571
62,761,2026-08-20 08:32:09.844,2026-08-20 13:55:39.558,955
63,772,2026-08-29 08:48:09.403,2026-08-29 14:12:32.111,451
64,773,2026-08-30 04:26:58.892,2026-08-30 09:51:31.604,1900


In [ ]:
result = Fido.search(
    a.Instrument.cryo_nirsp,
    a.Wavelength(1074*u.nm),
)

In [17]:
cryo_df = result[0].to_pandas()

In [21]:
cryo_df["Stokes Parameters"]

0      IQUV
1      IQUV
2         I
3         I
4         I
       ... 
293    IQUV
294    IQUV
295    IQUV
296    IQUV
297    IQUV
Name: Stokes Parameters, Length: 298, dtype: object

In [22]:
cryo_df_pol = cryo_df[cryo_df["Stokes Parameters"] == "IQUV"]

In [26]:
margin = pd.Timedelta("2h")
matches = []

for ii, aspiics_window in windows.iterrows():

    t0 = aspiics_window["start"] - margin
    t1 = aspiics_window["end"] + margin

    # interval overlap:
    # DKIST starts before search window ends
    # AND DKIST ends after search window starts
    mask = (
        (cryo_df_pol["Start Time"] <= t1) &
        (cryo_df_pol["End Time"] >= t0)
    )

    tmp = cryo_df_pol.loc[mask].copy()

    if len(tmp) == 0:
        continue

    tmp["aspiics_index"] = ii
    tmp["aspiics_start"] = aspiics_window["start"]
    tmp["aspiics_end"] = aspiics_window["end"]
    tmp["aspiics_orbit"] = aspiics_window["orbit_id"]

    matches.append(tmp)

matches = pd.concat(matches, ignore_index=True)

In [29]:
matches

,Product ID,Dataset ID,Start Time,End Time,Instrument,Wavelength,Status,Bounding Box,Movie Filename,Storage Bucket,...,Header Specification Version,Header Documentation URL,Info URL,Calibration Documentation URL,Spectral Lines,Experiment Description,aspiics_index,aspiics_start,aspiics_end,aspiics_orbit
0,L1-FCRUW,KUQGTD,2025-08-12 20:09:20.712500,2025-08-12 20:33:25.305601,CRYO-NIRSP,"[1072.6361661440012, 1076.5681944478374]",ACTIVE,"(1137.89,-54.54),(932.75,-310.56)",pid_3_66/KUQGTD/KUQGTD_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.edu/projects/cryo-nirsp...,[Fe XIII (1074.7 nm)],"The solar wind, a supersonic stream of charged...",8,2025-08-12 17:33:24.567,2025-08-12 20:16:29.571,306
1,L1-IGDWS,QOZIJQ,2025-08-12 21:41:41.832500,2025-08-12 22:05:46.425601,CRYO-NIRSP,"[1072.6361670288354, 1076.5681942212116]",ACTIVE,"(1137.94,-54.52),(932.73,-310.59)",pid_3_66/QOZIJQ/QOZIJQ_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.edu/projects/cryo-nirsp...,[Fe XIII (1074.7 nm)],"The solar wind, a supersonic stream of charged...",8,2025-08-12 17:33:24.567,2025-08-12 20:16:29.571,306
2,L1-POAHU,JKZKIO,2025-08-12 20:40:07.761500,2025-08-12 21:04:12.354601,CRYO-NIRSP,"[1072.6361637257091, 1076.5681973929552]",ACTIVE,"(1137.9,-54.53),(932.75,-310.57)",pid_3_66/JKZKIO/JKZKIO_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.edu/projects/cryo-nirsp...,[Fe XIII (1074.7 nm)],"The solar wind, a supersonic stream of charged...",8,2025-08-12 17:33:24.567,2025-08-12 20:16:29.571,306
3,L1-TIVBC,FATPWI,2025-08-12 21:10:54.806500,2025-08-12 21:34:59.399601,CRYO-NIRSP,"[1072.6347675015472, 1076.5698284770717]",ACTIVE,"(1137.92,-54.52),(932.74,-310.58)",pid_3_66/FATPWI/FATPWI_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.edu/projects/cryo-nirsp...,[Fe XIII (1074.7 nm)],"The solar wind, a supersonic stream of charged...",8,2025-08-12 17:33:24.567,2025-08-12 20:16:29.571,306
4,L1-XUJEZ,AJYOGG,2025-08-12 19:36:53.054500,2025-08-12 20:00:57.647601,CRYO-NIRSP,"[1072.6413109511864, 1076.580032738934]",ACTIVE,"(1137.9,-54.57),(932.73,-310.52)",pid_3_66/AJYOGG/AJYOGG_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.edu/projects/cryo-nirsp...,[Fe XIII (1074.7 nm)],"The solar wind, a supersonic stream of charged...",8,2025-08-12 17:33:24.567,2025-08-12 20:16:29.571,306
5,L1-UKAJY,YZKTFH,2025-08-12 18:32:56.250500,2025-08-12 19:30:16.066550,CRYO-NIRSP,"[1072.6413109270804, 1076.5800327653249]",ACTIVE,"(1138.83,-54.58),(932.74,-310.68)",pid_3_66/YZKTFH/YZKTFH_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.edu/projects/cryo-nirsp...,[Fe XIII (1074.7 nm)],"The solar wind, a supersonic stream of charged...",8,2025-08-12 17:33:24.567,2025-08-12 20:16:29.571,306
6,L1-SLYFN,UZDNYO,2025-09-17 20:19:40.236500,2025-09-17 22:22:19.240467,CRYO-NIRSP,"[1072.5924765894026, 1076.5279097910964]",ACTIVE,"(1066.61,795.01),(780.56,498.41)",pid_3_68/UZDNYO/UZDNYO_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.edu/projects/cryo-nirsp...,[Fe XIII (1074.7 nm)],Solar coronal magnetic fields play a key role ...,25,2025-09-17 16:00:49.063,2025-09-17 21:05:23.766,350
7,L1-AWFBG,AIOMGG,2025-10-05 21:16:48.007500,2025-10-05 23:47:46.097425,CRYO-NIRSP,"[1072.6247135346089, 1076.558504815405]",ACTIVE,"(974.33,-536.31),(682.23,-834.16)",pid_3_93/AIOMGG/AIOMGG_browse_movie.mp4,public,...,4.28.0,https://docs.dkist.nso.edu/projects/data-produ...,https://docs.dkist.nso.edu,https://docs.dkist.nso.e